In [1]:
# %load_ext autoreload
# %autoreload 2


import random

### My libraries (not pip)

try:
    import mathlib as Mathlib   #< c++ version (much faster training). Must be compiled first
    from NeuralNetwork_CPP import Layer, Batch, ActivationType
    import Activation
    print("Using C++ compiled mathlib!")
except ImportError:
    import Mathlib
    import Activation
    from Layer import Layer
    from Batch import Batch
    print("Using python mathlib as the C++ compiled library is not avaialble!")

import Loss
import Optimizer
import Accuracy


# import importlib
# importlib.reload(Mathlib)    #< cpp library
# importlib.reload(Activation)
# importlib.reload(Loss)
# importlib.reload(Optimizer)
# importlib.reload(Accuracy)
# importlib.reload(Layer)
# importlib.reload(Batch)


Using C++ compiled mathlib!


#### Go to `proofs_math/ActivationFuncs` for all Activation function proofs
<div style="display: flex; align-items: flex-start; gap: 20px;">
  <div>
    <img src="./proofs_math/ActivationFuncs/SoftmaxDerivative.jpg?version=1"
         alt="Softmax Proof"
         width="500">
  </div>

  <div style="display: flex; flex-direction: column; gap: 20px;">
    <img src="./proofs_math/ActivationFuncs/ReLUDerivative.jpg?version=1"
         alt="ReLU Proof"
         width="500">
  </div>

</div>



#### Go to `proofs_math/Loss` for all Loss proofs
<div style="display: flex; align-items: flex-start; gap: 20px;">
  <div>
    <img src="./proofs_math/Loss/SoftmaxCrossEntropyDerivative.jpg?version=1"
         alt="Neuron Proof"
         width="500">
  </div>
  <div style="display: flex; flex-direction: column; gap: 20px;">
    <img src="./proofs_math/Loss/LossDerivative.jpg?version=1"
         alt="Neuron Proof"
         width="500">
  </div>
</div>

#### Go to `proofs_math/Neuron` for all Neuron proofs
<div style="display: flex; align-items: flex-start; gap: 20px;">
  <div>
    <img src="./proofs_math/Neuron/NeuronDerivative.jpg?version=1"
         alt="Neuron Proof"
         width="500">
  </div>
</div>



## Functions to run a forward and backward pass on a super simple network with two layer totaling 9 neurons

In [11]:
random.seed(1)

def createInputs(inputCount=10):
    return([Mathlib.randomNumber(-10, 10, 2) for _ in range(inputCount)])

inputsCount = 5
target = 2
inputs = createInputs(inputsCount)

layer1NeuronCount = 4
layer2NeuronCount = 5

# layer1 = Layer(inputsCount, layer1NeuronCount, activationFunc=Activation.ReLU)         #< weights/biases determine neuron count
# layer2 = Layer(layer1NeuronCount, layer2NeuronCount, activationFunc=Activation.Pass)   #< output layer returns raw logits for softmax


layer1 = Layer(inputsCount, layer1NeuronCount, ActivationType.RELU)         #< weights/biases determine neuron count
layer2 = Layer(layer1NeuronCount, layer2NeuronCount, ActivationType.PASS)   #< output layer returns raw logits for softmax

accuracyCalculator = Accuracy.Accuracy_hard()

def stepForward():
    layer1Output = layer1.forward(inputs)
    layer2Output = layer2.forward(layer1Output)
    softmaxOutput = Activation.ProtectedSoftmax.forward(layer2Output)
    error = Loss.Entropy_sparse.forward(softmaxOutput, target)
    accuracy = accuracyCalculator.epoch(softmaxOutput, target)    #< we want to maximize the 2nd output
    return(layer1Output, layer2Output, softmaxOutput, error, accuracy)

def stepBackward(softmaxOutput):
    d_error  = Loss.Entropy_sparse.backward(softmaxOutput, target)
    d_softmax = Activation.Softmax.backward(softmaxOutput)
    d_crossEntropy = Mathlib.dotVectorMatrix(d_error, d_softmax) #< not that we can calculate the cross entropy function much simpler using the shortcut backwardCrossEntropy, but I choose to take the long way in this example because its easier to understand
    d_layer2 = layer2.backward(d_crossEntropy)
    d_layer1 = layer1.backward(d_layer2)
    return(d_error, d_softmax, d_crossEntropy, d_layer2, d_layer1)

def printForwardSummary(layer1Output, layer2Output, softmaxOutput, error, accuracy):
    print(f"INPUTS: {inputs}\n    This layer should have {inputsCount} inputs")
    print(f"WEIGHTS: {layer1.getWeights()}\n    this layer should have {layer1NeuronCount} weights with {inputsCount} values in them")
    print(f"BIASES: {layer1.getBiases()}\n    this layer should have {layer1NeuronCount} biases")
    print(f"RESULT: {layer1Output}\n     this layer should have {layer1NeuronCount} outputs")
    print()
    print(f"WEIGHTS: {layer2.getWeights()}\n    this layer should have {layer2NeuronCount} weights with {len(layer1Output)} values in them")
    print(f"BIASES: {layer2.getBiases()}\n    this layer should have {layer2NeuronCount} biases")
    print(f"RESULT: {layer2Output}\n     this layer should have {layer2NeuronCount} outputs")
    print("SOFTMAX: ", softmaxOutput)
    print("TARGET: ", target)
    print(f"ERROR: {error} should be between 0 and 16.12")
    print("ACCURACY: ", accuracy)

def printBackwardSummary(d_error, d_softmax, d_crossEntropy, d_layer2, d_layer1):
    print("d_ERROR: ", d_error)
    print(f"d_SOFTMAX: {d_softmax}\n    this layer should be {layer2NeuronCount}x{len(d_softmax)}")
    print(f"d_CROSS_ENTROPY: {d_crossEntropy}\n    this layer should have {layer2NeuronCount} outputs")
    print("d_LAYER 2: ", d_layer2)
    print("d_LAYER 1: ", d_layer1)

layer1Output, layer2Output, softmaxOutput, error, accuracy = stepForward()
printForwardSummary(layer1Output, layer2Output, softmaxOutput, error, accuracy)
d_error, d_softmax, d_crossEntropy, d_layer2, d_layer1 = stepBackward(softmaxOutput)
printBackwardSummary(d_error, d_softmax, d_crossEntropy, d_layer2, d_layer1)

INPUTS: [-6.9, 7.54, -8.13, 6.96, -4.36]
    This layer should have 5 inputs
WEIGHTS: [[0.51, 0.12, -0.26, 0.56, -0.28], [0.86, 0.44, -0.74, 0.61, -0.81], [-0.43, 0.18, -0.11, 0.35, 0.22], [0.16, -0.23, -0.6, 0.35, -0.76]]
    this layer should have 4 weights with 5 values in them
BIASES: [0.0, 0.0, 0.0, 0.0]
    this layer should have 4 biases
RESULT: [4.618, 11.177000000000001, 6.6953000000000005, 7.7894]
     this layer should have 4 outputs

WEIGHTS: [[-0.11, -0.76, 0.57, 0.47], [0.52, 0.74, 0.79, -0.77], [-0.62, -0.27, 0.79, 0.66], [0.82, 0.13, 0.97, 0.38], [0.58, 0.41, -0.83, 0.69]]
    this layer should have 5 weights with 4 values in them
BIASES: [0.0, 0.0, 0.0, 0.0, 0.0]
    this layer should have 5 biases
RESULT: [-1.5251610000000015, 9.963789000000004, 4.549341, 14.694183, 7.078597]
     this layer should have 5 outputs
SOFTMAX:  [8.953326660715414e-08, 0.008741219337096741, 3.891411238858444e-05, 0.9907316317837651, 0.0004881452334828891]
TARGET:  2
ERROR: 10.15415358679199

### Optimizing the neural network

In [12]:
optimizer = Optimizer.Optimizer_SGD(learning_rate=0.01)
for i in range(5000):
    layer1Output, layer2Output, softmaxOutput, error, accuracy = stepForward()
    d_error, d_softmax, d_crossEntropy, d_layer2, d_layer1 = stepBackward(softmaxOutput)
    #printForwardSummary(layer1Output, layer2Output, softmaxOutput, error, accuracy)
    #printBackwardSummary(d_error, d_softmax, d_crossEntropy, d_layer2, d_layer1)
    optimizer.update(layer2)
    optimizer.update(layer1)
    if (i%100) == 0:
        print(f"Epoch {i}: Error: {error}, accuracy: {accuracy}")

Epoch 0: Error: 10.154153586791994, accuracy: 0.0
Epoch 100: Error: 16.11809565095832, accuracy: 0.0
Epoch 200: Error: 16.11809565095832, accuracy: 0.0
Epoch 300: Error: 16.11809565095832, accuracy: 0.0
Epoch 400: Error: 16.11809565095832, accuracy: 0.0
Epoch 500: Error: 16.11809565095832, accuracy: 0.0
Epoch 600: Error: nan, accuracy: 0.0
Epoch 700: Error: nan, accuracy: 0.0
Epoch 800: Error: nan, accuracy: 0.0
Epoch 900: Error: nan, accuracy: 0.0
Epoch 1000: Error: nan, accuracy: 0.0
Epoch 1100: Error: nan, accuracy: 0.0
Epoch 1200: Error: nan, accuracy: 0.0
Epoch 1300: Error: nan, accuracy: 0.0
Epoch 1400: Error: nan, accuracy: 0.0
Epoch 1500: Error: nan, accuracy: 0.0
Epoch 1600: Error: nan, accuracy: 0.0
Epoch 1700: Error: nan, accuracy: 0.0
Epoch 1800: Error: nan, accuracy: 0.0
Epoch 1900: Error: nan, accuracy: 0.0
Epoch 2000: Error: nan, accuracy: 0.0
Epoch 2100: Error: nan, accuracy: 0.0
Epoch 2200: Error: nan, accuracy: 0.0
Epoch 2300: Error: nan, accuracy: 0.0
Epoch 2400: Err

In [2]:
import json
import time
start = time.time()
BATCH_SIZE = int(1000/20)#32

with open("datasets/squareData.json", "r") as f:
    a = json.load(f)
    X = a["X"]         #< 32x32 image 1 = pixel, 0 = no pixel
    y = a["y"]

print("Converting data to hilbert space")
for i in range(len(X)):
    X[i] = Mathlib.hilbertFlatten(X[i])   #< map to 1D using hilbert space (hilbert is used here so the resolution has minimal effect)
    print(f"{i}/{len(X)}", end="\r")
print(f"\rDone/{len(X)}")

spiralLayer1 = Batch(BATCH_SIZE, 32*32, 32, ActivationType.LEAKY_RELU)
spiralLayer2 = Batch(BATCH_SIZE, 32, 2, ActivationType.PASS)

try:
    with open("trainedModel.json", "r") as f:
        savedModel = json.load(f)
    spiralLayer1.setWeights(savedModel["layer1"]["weights"])
    spiralLayer1.setBiases(savedModel["layer1"]["biases"])
    spiralLayer2.setWeights(savedModel["layer2"]["weights"])
    spiralLayer2.setBiases(savedModel["layer2"]["biases"])
    print("Successfully loaded 'trainedModel.json'!")

except FileNotFoundError:
    print("Error: 'trainedModel.json' not found. Training from scratch.")

# softmaxCrossEntropy = Loss.SoftmaxCrossEntropy()
softmaxCrossEntropy = Loss.SoftmaxCrossEntropy_batch()
spiralOptimizer = Optimizer.Optimizer_SGD(learning_rate=0.01)
accuracyCalculator = Accuracy.Accuracy_hard()

for epoch in range(1_000):
    numSamples = len(X)

    for i in range(0, numSamples, BATCH_SIZE):
        X_batch = X[i : i + BATCH_SIZE]
        y_batch = y[i : i + BATCH_SIZE]

        ### Forwards pass
        ### layer1 -> layer2 -> layer3 -> softmax_error
        layer1Output = spiralLayer1.forward(X_batch)                              #< returns a list of lists of outputs
        layer2Output = spiralLayer2.forward(layer1Output)                         #< returns a list of lists of outputs
        # print(layer2Output)
        error = softmaxCrossEntropy.forward(layer2Output, y_batch)                #< returns mean error
        accuracy = accuracyCalculator.epoch(layer2Output, y_batch)                #< returns accuracy of model %right

        ### Backwards pass
        ### error_softmax -> layer2 ->  layer1 
        d_crossEntropy = softmaxCrossEntropy.backward()
        d_layer2 = spiralLayer2.backward(d_crossEntropy)
        # print([Mathlib.mean(d) for d in d_layer2])
        d_layer1 = spiralLayer1.backward(d_layer2)
        # print([Mathlib.mean(d) for d in d_layer1])

        spiralOptimizer.update(spiralLayer2)
        spiralOptimizer.update(spiralLayer1)

    print(f"Epoch {epoch}: Error: {error}, accuracy: {accuracy}")  #< Normal to bounce around at the start
    
    ## Save model
    modelState = {
        "layer1": {
            "weights": spiralLayer1.getWeights(),
            "biases": spiralLayer1.getBiases()
        },
        "layer2": {
            "weights": spiralLayer2.getWeights(),
            "biases": spiralLayer2.getBiases()
        }
    }

    if epoch%10 == 0:
        with open("trainedModel.json", "w") as f:
            json.dump(modelState, f, indent=4)

end = time.time()
print(f"Training took: {end-start} seconds")


Converting data to hilbert space
Done/1000
Error: 'trainedModel.json' not found. Training from scratch.
Epoch 0: Error: 0.7118792059763069, accuracy: 0.506
Epoch 1: Error: 0.710254857821917, accuracy: 0.5175
Epoch 2: Error: 0.7072559449967014, accuracy: 0.5253333333333333
Epoch 3: Error: 0.7049082446472926, accuracy: 0.531
Epoch 4: Error: 0.702835135371106, accuracy: 0.5348
Epoch 5: Error: 0.7011776308217824, accuracy: 0.5383333333333333
Epoch 6: Error: 0.6996625690975782, accuracy: 0.5418571428571428
Epoch 7: Error: 0.6979727275929021, accuracy: 0.54475
Epoch 8: Error: 0.6966049757988486, accuracy: 0.5476666666666666
Epoch 9: Error: 0.6948826658769943, accuracy: 0.5498
Epoch 10: Error: 0.6930084273009198, accuracy: 0.5523636363636364
Epoch 11: Error: 0.6912933817224549, accuracy: 0.5546666666666666
Epoch 12: Error: 0.6893203842441069, accuracy: 0.5566153846153846
Epoch 13: Error: 0.6880739232217354, accuracy: 0.5587857142857143
Epoch 14: Error: 0.686145214821144, accuracy: 0.561133333

The raw Python model takes about 45 minutes to train to 90% and slightly less than 90 minutes to fully train.

The C++ model takes about 2 minutes to train to 90% accuracy and slighly more than 4 minutes to fully train Thats 21x the speed!

In [ ]:
import Demo
import tkinter as tk
root = tk.Tk()
app = Demo.DrawingApp(root)
root.mainloop()